# ASA Perception Evaluation Runs


In [ ]:
# ASA Imports
# Notebook Specifics / Temporary Functions
#
import logging

from asa._tools.custom_logging import setup_logging
from asa._tools.paths import repo_root

log = logging.getLogger(f"{__name__}.app")
setup_logging(level="DEBUG")

DATA_IN = repo_root() / "data_in"

log.info("Notebook Perception Evaluation")

## Establish Text Benchmarks & Function To Simulate Utterances for An Input Stream

In [37]:
# Benchmark Text Datasets Use For Evaluation
#

import pandas as pd

BENCH_SIMPLE = DATA_IN / "bench_simple_ekman6.parquet"
BENCH_BRIGHTER = DATA_IN / "bench_brighter_eng.parquet"

simple_bench = pd.read_parquet(BENCH_SIMPLE)
brighter_bench = pd.read_parquet(BENCH_BRIGHTER)

display(simple_bench.attrs)
display(simple_bench.shape)
display(simple_bench.head(10))

display(brighter_bench.attrs)
display(brighter_bench.shape)
display(brighter_bench.head(10))



{'corpus': 'simple-ekman6',
 'representation': 'ekman6/1',
 'axis_map': {},
 'unannotated_axes': [],
 'rows': 20}

(20, 10)

,source,split,id,text,anger,disgust,fear,happiness,sadness,surprise
0,simple-ekman6,all,simple-ekman6_00000,I am so happy,0.0,0.0,0.0,1.0,0.0,0.0
1,simple-ekman6,all,simple-ekman6_00001,I'm absolutely delighted,0.0,0.0,0.0,1.0,0.0,0.0
2,simple-ekman6,all,simple-ekman6_00002,I was gutted,0.0,0.0,0.0,0.0,1.0,0.0
3,simple-ekman6,all,simple-ekman6_00003,I feel miserable today,0.0,0.0,0.0,0.0,1.0,0.0
4,simple-ekman6,all,simple-ekman6_00004,that is absolutely revolting,0.0,1.0,0.0,0.0,0.0,0.0
5,simple-ekman6,all,simple-ekman6_00005,I'm repulsed by it,0.0,1.0,0.0,0.0,0.0,0.0
6,simple-ekman6,all,simple-ekman6_00006,I'm furious about it,1.0,0.0,0.0,0.0,0.0,0.0
7,simple-ekman6,all,simple-ekman6_00007,that made me really angry,1.0,0.0,0.0,0.0,0.0,0.0
8,simple-ekman6,all,simple-ekman6_00008,I was terrified,0.0,0.0,1.0,0.0,0.0,0.0
9,simple-ekman6,all,simple-ekman6_00009,I'm scared of what happens next,0.0,0.0,1.0,0.0,0.0,0.0


{'corpus': 'brighter-eng',
 'representation': 'ekman6/1',
 'axis_map': {'joy': 'happiness'},
 'unannotated_axes': ['disgust'],
 'rows': 8522}

(8522, 10)

,source,split,id,text,anger,disgust,fear,happiness,sadness,surprise
0,brighter-eng,train,eng_train_track_a_00001,"colorado, middle of nowhere.",0.0,NaN,1.0,0.0,0.0,1.0
1,brighter-eng,train,eng_train_track_a_00002,this involved swimming a pretty large lake tha...,0.0,NaN,1.0,0.0,0.0,0.0
2,brighter-eng,train,eng_train_track_a_00003,it was one of my most shameful experiences.,0.0,NaN,1.0,0.0,1.0,0.0
3,brighter-eng,train,eng_train_track_a_00004,"after all, i had vegetables coming out my ears...",0.0,NaN,0.0,0.0,0.0,0.0
4,brighter-eng,train,eng_train_track_a_00005,then the screaming started.,0.0,NaN,1.0,0.0,1.0,1.0
5,brighter-eng,train,eng_train_track_a_00006,"they don't fear death, and it seems they belie...",0.0,NaN,1.0,0.0,0.0,1.0
6,brighter-eng,train,eng_train_track_a_00007,you know what happens when i get one of these ...,1.0,NaN,1.0,0.0,0.0,0.0
7,brighter-eng,train,eng_train_track_a_00008,my stomach even started giving me fits.,0.0,NaN,1.0,0.0,1.0,0.0
8,brighter-eng,train,eng_train_track_a_00009,"well, as we're bowling, my dinner began to not...",0.0,NaN,1.0,0.0,0.0,0.0
9,brighter-eng,train,eng_train_track_a_00010,hondas are notoriously great cars for long tri...,0.0,NaN,0.0,1.0,0.0,0.0


---
## ASA Perception & Decode Runs Setup


In [ ]:
# Establish Different Lexicons
#

import json

from asa.affect_model.belief import AffectModel
from asa.affect_model.folding import Assign, ConfidenceWeighted
from asa.core.affect import Target, utc_now
from asa.core.representations import EKMAN6, PLUTCHIK8
from asa.perception.decode_keyword import EKMAN6_KEYWORDS, LEXICON_HANDWRITTEN, KeywordDecoder
from asa.perception.nrc_eil import load_tables

EIL_LEXICON_SOURCE = DATA_IN / "NRC-Emotion-Intensity-Lexicon-v1.loaded-2026-08-13.json"

# Check Attributes
print("Attributes")
written = json.loads(EIL_LEXICON_SOURCE.read_text(encoding="utf-8"))
for key, value in sorted(written.items()):
    if not isinstance(value, (dict, list)):
        print(f"{key:12}: {value}")
entries = written.get("entries", {})

print("id present  :", "id" in written)
print(f"{'entries':12}: {len(written.get('entries', {}))} emotions")
print("emotions    :", sorted(entries))

# Get the lexicon data
lex = load_tables(EIL_LEXICON_SOURCE, [PLUTCHIK8, EKMAN6])
print("\nLoader")
print(f"source      : {lex.source}")
for rep_id, table in lex.tables.items():
    entries = sum(len(words) for words in table.values())
    distinct = len({word for words in table.values() for word in words})
    print(f"{rep_id:<12} {len(table)} axes, {entries:,} entries, {distinct:,} distinct words")

table_hand, lexicon_hand = EKMAN6_KEYWORDS, LEXICON_HANDWRITTEN
table_ek, lexicon = lex.tables[EKMAN6.id], lex.source
table_pk, lexicon = lex.tables[PLUTCHIK8.id], lex.source



In [ ]:
# Establish Decoder(s)
#

# Decoder(s) for Testing
decoder1 = KeywordDecoder(representation=EKMAN6, table=table_hand, lexicon=lexicon_hand)
decoder2 = KeywordDecoder(representation=PLUTCHIK8, table=table_hand, lexicon=lexicon_hand) # Should fail?
decoder3 = KeywordDecoder(representation=PLUTCHIK8, table=table_ek, lexicon=lexicon) # Should fail?
decoder4 = KeywordDecoder(representation=EKMAN6, table=table_ek, lexicon=lexicon)
decoder5 = KeywordDecoder(representation=PLUTCHIK8, table=table_pk, lexicon=lexicon) # Should fail?
# decoder6 = KeywordDecoder(representation=EKMAN6, table=table_pk, lexicon=lexicon) # Should fail?


In [ ]:
# Enables Console Input in Notebook
#

def read_or_end(prompt: str) -> str:
    """Notebook stand-in for Ctrl-D — no frontend here can send a real EOF."""
    text = input(prompt)
    if text.strip() == ":q":
        raise EOFError
    return text

In [ ]:
# An input source that replays a benchmark frame as utterances carrying ground truth
# For evaluating the ASA pipeline

import asyncio
from collections.abc import AsyncIterator, Iterable

import pandas as pd

from asa.core.affect import AffectVector, Utterance
from asa.core.representations import AffectRepresentation


class UtterancesFromDF:
    """Replays a benchmark frame as Utterances carrying their intended affect.

    Knows nothing about any particular corpus: it reads ``text`` and the axis columns of the
    representation it is given, which is exactly what the standard benchmark shape
    guarantees are there.

    ``intended`` is the ground truth a decoder is scored against. ``Utterance``'s own
    docstring names a labelled benchmark set as one of only two sources entitled to set it,
    so this is the sanctioned route rather than a test fixture leaking into the runtime.

    ``assume_absent`` names axes whose corpus does not annotate them; any *other* unlabelled
    axis raises. The frames keep an unannotated axis as NaN because ``rest`` is not
    "unknown" — it is the positive claim that the emotion is absent — so somebody has to
    make that claim, and it should be whoever runs the replay, out loud. BRIGHTER-eng needs
    ``assume_absent=("disgust",)``; its ``attrs`` says so.

    ``gap`` is what makes this a *source* rather than a list. A real participant speaks with
    pauses, and an async generator that never awaits would submit the whole frame before the
    consumer ran once — the published order would then be an artefact of the fake rather
    than the pipeline's behaviour.
    """

    def __init__(self,
                 source_df: pd.DataFrame,
                 representation: AffectRepresentation,
                 *,
                 source: str = "input:eval_replay",
                 gap: float = 1.0,
                 assume_absent: Iterable[str] = ()) -> None:
        self._source_df = source_df
        self._rep = representation
        self._source = source
        self._gap = gap
        self._absent = {str(axis) for axis in assume_absent}

    def _intended(self, row: pd.Series) -> AffectVector:
        """The row's labels as a vector; an unmeasured axis is refused unless assumed absent."""
        values = dict(self._rep.rest_vector().values)
        for axis in self._rep.axes:
            magnitude = row[axis]
            if pd.isna(magnitude):
                if str(axis) not in self._absent:
                    raise ValueError(f"{axis} is unlabelled in row {row['id']!r} — name it in "
                                     f"assume_absent to assert the corpus means it is absent")
                continue                  # rest_vector's seeded value stands
            values[str(axis)] = float(magnitude)
        return AffectVector(representation=self._rep.id, values=values)

    async def events(self) -> AsyncIterator[Utterance]:
        for _, row in self._source_df.iterrows():
            yield Utterance(text=str(row["text"]),
                            source=self._source,
                            intended=self._intended(row))
            await asyncio.sleep(self._gap)

In [ ]:
# Stand In for Observer Implementation - Basic Logging
#

from asa.core.affect import AffectEvidence, AffectState, AffectVector, Utterance
from asa.core.observers import Event


def axes(vector: AffectVector | None) -> dict[str, float] | None:
    """Non-rest axes only, rounded — a full eight-axis dict at full precision is unreadable.

    **`None` in, `None` out, and not an empty dict.** A source that does not declare its intent is
    making no claim; an empty dict is the claim that every axis is at rest. `Utterance.intended` is
    `None` for a person typing at a console and populated for a generated or benchmark source, so
    collapsing the two would report ground truth where there is none — the unmeasured-versus-absent
    mistake that keeps BRIGHTER's disgust as NaN rather than zero-filling it.

    Three decimals rather than two so that decay stays visible between folds: the difference
    between a belief of 0.625 and one that has aged to 0.624 is the whole point of watching a
    replay, and two decimals hides it.

    Filters against zero rather than the representation's `rest`. They coincide for all three
    declared representations, and this is a debug log rather than a record — but a representation
    resting off zero would show every axis, so do not carry this into the recorder.
    """
    if vector is None:
        return None
    return {str(k): round(magnitude, 3) for k, magnitude in vector.values.items() if magnitude}


def clipped(text: str | None, width: int = 60) -> str:
    """A rationale short enough to keep the columns aligned.

    The log is for *watching* a run; the Collector keeps whole records for measuring it, so
    nothing is lost by truncating here. Over BRIGHTER-eng the median rationale is 39 characters
    and the tail reaches 265 — long enough to push every following column off the line.
    """
    if not text:
        return ""
    return text if len(text) <= width else f"{text[:width - 1]}…"

def log_event(event: Event) -> None:
    """Temporary stand-in for the recorder — schema, time, id, then the payload.

    Every branch keeps the same first three columns so the lines read as a table. A state has no
    id column of its own: it is the belief at an instant rather than a record pointing at another.

    The source column is 24 wide because `%-Ns` pads but never truncates, so anything longer
    shunts every following column right: `decoder:rule:handwritten` is exactly 24, and 18 —
    what this was — misaligned every evidence line the moment a lexicon name joined the source.
    """
    when = event.at.strftime("%H:%M:%S.%f")[:-3]

    if isinstance(event, Utterance):
        log.debug("%-12s %s  %s  %-5s %-24s %r intended=%s",
                  event.schema, when, event.id, "", event.source, event.text,
                  axes(event.intended))
    elif isinstance(event, AffectEvidence):
        log.debug("%-12s %s  %s  %-5s %-24s %s  conf=%s  %s",
                  event.schema, when, event.of_input, event.target, event.source,
                  axes(event.affect), event.confidence, clipped(event.rationale))
    elif isinstance(event, AffectState):
        log.debug("%-12s %s  %-12s  %-5s %-24s other=%s self=%s expressed=%s",
                  event.schema, when, "", "", "", axes(event.other), axes(event.self_),
                  axes(event.expressed))
    else:
        log.debug("%-12s %s  %r", event.schema, when, event)

    # Full dataclass
    # log.debug("%-12s %s", event.schema, event)


In [ ]:
# Collector - keeps whole records for measuring a run
#

class Collector:
    """Keeps every published record, split by type — the notebook's stand-in for the recorder.

    Registered *alongside* `log_event` because the two do different jobs: the log is for
    watching a run, this is for measuring it. It keeps whole records rather than extracted
    fields, since analysis wants full precision and every axis — both of which `log_event`
    deliberately discards.
    """

    def __init__(self) -> None:
        self.utterances: list[Utterance] = []
        self.evidence: list[AffectEvidence] = []
        self.states: list[AffectState] = []

    def __call__(self, event: Event) -> None:
        if isinstance(event, Utterance):
            self.utterances.append(event)
        elif isinstance(event, AffectEvidence):
            self.evidence.append(event)
        elif isinstance(event, AffectState):
            self.states.append(event)


def to_frame(collector: Collector, rep: AffectRepresentation, *,
             unmeasured: Iterable[str] = ()) -> pd.DataFrame:
    """One row per decoded utterance: ground truth beside what the decoder made of it.

    Joined on `of_input`, which is the only link between the two streams — a state carries no
    id and is therefore not joined here.

    **`unmeasured` restores what `assume_absent` collapsed.** The benchmark frames keep an
    unannotated axis as NaN because `rest` is not "unknown", but `UtterancesFromDF._intended`
    has to turn it into a real magnitude to build a vector, so by the time a record exists the
    distinction is gone. Pass the corpus's own `attrs["unannotated_axes"]` and those columns go
    back to NaN — without it, every disgust prediction against BRIGHTER-eng scores as a false
    positive on a class nobody labelled.
    """
    axes = [str(axis) for axis in rep.axes]
    unmeasured = {str(axis) for axis in unmeasured}
    by_id = {utterance.id: utterance for utterance in collector.utterances}

    rows = []
    for evidence in collector.evidence:
        if evidence.target is not Target.OTHER:      # only the decoder's readings
            continue
        utterance = by_id.get(evidence.of_input)
        truth = utterance.intended if utterance and utterance.intended else None
        row = {"id": evidence.of_input,
               "text": utterance.text if utterance else None,
               "decoder": evidence.source,
               "confidence": evidence.confidence,
               "rationale": evidence.rationale}
        for axis in axes:
            row[f"true_{axis}"] = (float("nan") if truth is None or axis in unmeasured
                                   else truth.values[axis])
            row[f"pred_{axis}"] = evidence.affect.values[axis]
        row["true_axes"] = "" if truth is None else " ".join(
            axis for axis in axes if axis not in unmeasured and truth.values[axis] > rep.rest)
        row["pred_axes"] = " ".join(axis for axis in axes if evidence.affect.values[axis] > rep.rest)
        rows.append(row)

    return pd.DataFrame(rows).set_index("id")



In [ ]:
# Setup before each run
#

from asa.core.observers import Observers


def new_run(rep: AffectRepresentation = EKMAN6, *,
            unstated_confidence: float = 0.5, 
            half_life_s: float = 45.0,
            watch: bool = True):
    """Fresh wiring for one run: observers, collector, and a model holding no prior belief.

    **A run is a fresh composition, not a continuation.** `Observers.register` is a plain
    append, so a run cell registering into a shared registry adds another observer every time
    it is re-run — the orphans stay registered and keep collecting. And `AffectModel` carries
    its belief forward, so a second run would fold its opening rows onto what the first left.

    `unstated_confidence` is a parameter here because it is the fold weight for every
    observation in a rule-decoder run — the decoder never states a confidence — so it is the
    one knob a sensitivity check needs to vary.
    """
    collector = Collector()
    observers = Observers()
    if watch:
        observers.register(log_event)
        
    observers.register(collector)
    model = AffectModel(representation=rep,
                        policies={Target.OTHER: ConfidenceWeighted(unstated_confidence=unstated_confidence,
                                                                   max_weight=1.0, 
                                                                   refresh_above=0.5),
                                  Target.SELF: Assign(),
                                  Target.EXPRESSED: Assign()},
                        half_lives={Target.OTHER: half_life_s, Target.SELF: half_life_s},
                        started_at=utc_now())
    
    return observers, collector, model


---
## ASA Perception & Decode Runs - Two Types of Run

In [ ]:
# Run the ASA Agent - Manual Text Console
# NB: In Notebook is asyncio
#

# from asa.core.observers import Observers
from asa.perception.text_console import TextConsole
from asa.runtime import run_agent

source = TextConsole(read=read_or_end)
test_decoder = decoder1
observers, collector, model = new_run()

await run_agent(source=source, decoder=test_decoder, state_writer=model.observe, observers=observers)


In [39]:
# Run the ASA Agent - Benchmark Text Input
# NB: In Notebook is asyncio
#


# decoder4 = KeywordDecoder(representation=EKMAN6, table=table_ek, lexicon=lexicon)
# decoder5 = KeywordDecoder(representation=PLUTCHIK8, table=table_pk, lexicon=lexicon)

from asa.runtime import run_agent

REP = EKMAN6
# REP = PLUTCHIK8

GAP = 0
# source_df = simple_bench.sample(10, random_state=0)
source_df = brighter_bench.sample(5000, random_state=0)

source = UtterancesFromDF(source_df=source_df, 
                          representation=REP, 
                          gap=GAP,
                          assume_absent=source_df.attrs["unannotated_axes"])
test_decoder = decoder4
observers, collector, model = new_run(watch=False)

await run_agent(source=source, decoder=test_decoder, state_writer=model.observe, observers=observers)



# Run analysis

In [16]:
def score(df: pd.DataFrame, rep: AffectRepresentation, *, threshold: float = 0.0) -> dict:
    """Micro-averaged precision, recall and F1 over the axis decisions in a `to_frame` result.

    **The frame says what is scoreable, so nothing has to be passed in twice.** `to_frame`
    already wrote NaN into the `true_` column of any axis the corpus never annotated, so a
    column that is entirely NaN is unscoreable by construction — no second `unmeasured`
    argument to fall out of step with the first.

    Rows carrying no ground truth at all are dropped rather than counted. A console utterance
    has `intended=None` and every `true_` column NaN; a benchmark row that is genuinely
    all-rest has 0.0. Those are different claims — no measurement against a measurement of
    absence — and scoring the first would count every prediction as a false positive.

    **Micro rather than macro**: every (row, axis) decision counts once, so a rare axis does
    not weigh as much as a common one. BRIGHTER's fear outnumbers its anger four to one, so
    report macro as well if the per-axis balance is the question.

    *threshold* is what counts as "fired", and it is the same knob as `load_tables`' `floor` —
    under the decoder's `max`, decoding at `floor=t` and thresholding a `floor=0` decode at `t`
    give identical magnitudes. Sweep it here rather than reloading the lexicon.
    """
    axes = [axis for axis in map(str, rep.axes) if not df[f"true_{axis}"].isna().all()]
    labelled = df[df[[f"true_{axis}" for axis in axes]].notna().any(axis=1)]

    # to_numpy() because the two column sets have different labels and pandas aligns on them —
    # `pred & true` as frames would union the labels and give all-NaN rather than an error.
    pred = (labelled[[f"pred_{axis}" for axis in axes]] > threshold).to_numpy()
    true = (labelled[[f"true_{axis}" for axis in axes]] > rep.rest).to_numpy()

    tp = int((pred & true).sum())
    fp = int((pred & ~true).sum())
    fn = int((~pred & true).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0

    return {"threshold": threshold, "scored_rows": len(labelled), "tp": tp, "fp": fp, "fn": fn,
            "precision": round(precision, 3), "recall": round(recall, 3),
            "f1": round(2 * precision * recall / (precision + recall), 3) if precision + recall else 0.0,
            "exact": int((pred == true).all(axis=1).sum()),
            "silent": int((~pred).all(axis=1).sum())}

In [40]:
# Records analysis post run
#

df = to_frame(collector, EKMAN6, unmeasured=brighter_bench.attrs["unannotated_axes"])
# df[["text", "true_axes", "pred_axes", "rationale"]]
display(f"{len(df)} rows decoded")
display(df.head(10))

score(df, EKMAN6)                                   
pd.DataFrame([score(df, EKMAN6, threshold=t)        
              for t in (0.0, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7)])


'5000 rows decoded'

,text,decoder,confidence,rationale,true_anger,pred_anger,true_disgust,pred_disgust,true_fear,pred_fear,true_happiness,pred_happiness,true_sadness,pred_sadness,true_surprise,pred_surprise,true_axes,pred_axes
id,,,,,,,,,,,,,,,,,,
e3f6da6b8800,my face turned red because i was afraid and i ...,decoder:rule:nrc-eil,None,matched: fear=afraid,0.0,0.000,NaN,0.000,1.0,0.844,0.0,0.000,0.0,0.000,0.0,0.000,fear,fear
28eb3a0fbdf8,""" which only served to make us laugh harder.",decoder:rule:nrc-eil,None,"matched: happiness=laugh, surprise=laugh",0.0,0.000,NaN,0.000,0.0,0.000,1.0,0.891,0.0,0.000,0.0,0.258,happiness,happiness surprise
ee47413774b2,immediately i regretted leaving behind my sung...,decoder:rule:nrc-eil,None,"matched: sadness=leaving, sadness=regretted",0.0,0.000,NaN,0.000,1.0,0.000,0.0,0.000,0.0,0.652,0.0,0.000,fear,sadness
fcd39fdecb1a,thinking that maybe devan had come in to use t...,decoder:rule:nrc-eil,None,"matched: sadness=alas, sadness=empty",0.0,0.000,NaN,0.000,0.0,0.000,0.0,0.000,0.0,0.364,0.0,0.000,,sadness
51ead81c2875,i still cant fucking believe it.,decoder:rule:nrc-eil,None,no keyword matched,1.0,0.000,NaN,0.000,1.0,0.000,0.0,0.000,0.0,0.000,1.0,0.000,anger fear surprise,
b9a839e90d07,every movement of the body below my waist caus...,decoder:rule:nrc-eil,None,"matched: fear=pain, sadness=pain",0.0,0.000,NaN,0.000,1.0,0.594,0.0,0.000,1.0,0.719,0.0,0.000,fear sadness,fear sadness
f10f0f1f12da,i can only remember my eye getting swollen by ...,decoder:rule:nrc-eil,None,"matched: disgust=bug, fear=bug, happiness=life",0.0,0.000,NaN,0.359,0.0,0.188,1.0,0.438,0.0,0.000,0.0,0.000,happiness,disgust fear happiness
21f5cbc9bc42,"i rolled my eyes, like there were rules to thi...",decoder:rule:nrc-eil,None,"matched: anger=damn, disgust=damn",0.0,0.735,NaN,0.359,0.0,0.000,0.0,0.000,0.0,0.000,0.0,0.000,,anger disgust
fb22feae05c2,"the stranger is left speechless, confused, and...",decoder:rule:nrc-eil,None,"matched: fear=stranger, happiness=proud",0.0,0.000,NaN,0.000,1.0,0.453,1.0,0.704,0.0,0.000,1.0,0.000,fear happiness surprise,fear happiness


,threshold,scored_rows,tp,fp,fn,precision,recall,f1,exact,silent
0,0.0,5000,3151,3766,4545,0.456,0.409,0.431,686,1645
1,0.2,5000,2970,3259,4726,0.477,0.386,0.427,776,1816
2,0.3,5000,2723,2615,4973,0.510,0.354,0.418,847,2158
3,0.4,5000,2519,2135,5177,0.541,0.327,0.408,917,2374
4,0.5,5000,2091,1499,5605,0.582,0.272,0.371,936,2842
5,0.6,5000,1504,797,6192,0.654,0.195,0.301,887,3518
6,0.7,5000,1038,409,6658,0.717,0.135,0.227,786,3971
